# Generación y guardado de patches desde OME-TIFF + GeoJSON
## TFM-Francesca Romana Val Bagli

Este notebook hace solo el preprocesado de casos y la exportación de patches. No entrena modelos y no carga checkpoints.

Flujo:

1. Empareja imágenes `.ome.tif/.ome.tiff/.tif/.tiff` con anotaciones `.geojson`.
2. Lee las anotaciones de QuPath.
3. Genera máscaras semánticas con clases `0`, `1`, `2` y `255`.
4. Extrae ROIs y patches.
5. Divide por caso en `train`, `val` y `test`.
6. Guarda los patches con la siguiente estructura:

```text
TFM_UNI_PATCHES_224/
├── train/
│   ├── images/
│   └── masks/
├── val/
│   ├── images/
│   └── masks/
├── test/
│   ├── images/
│   └── masks/
├── metadata.csv
└── split_case_ids.json
```


In [1]:

# ============================================================
# 1. IMPORTS
# ============================================================

from pathlib import Path
import json
import math
import random
import re
import shutil

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from tqdm import tqdm

import rasterio
from rasterio.windows import Window
from scipy import ndimage

import matplotlib.pyplot as plt
from IPython.display import display


In [2]:

# ============================================================
# 2. CONFIGURACIÓN PRINCIPAL
# ============================================================

# Carpeta base con las imágenes y los GeoJSON.
BASE_PATH = Path("TFM")
IMAGES_DIR = BASE_PATH / "imagenes"
GEOJSON_DIR = BASE_PATH / "geojsons"

# Tamaño de patch.
# Para 224 usa PATCH_SIZE=224 y STRIDE=112.
# Para 256 usa PATCH_SIZE=256 y STRIDE=128.
PATCH_SIZE = 224
STRIDE = 112

# Carpeta de salida.
EXPORT_DIR = Path(f"TFM_UNI_PATCHES_{PATCH_SIZE}")
OVERWRITE_EXPORT_DIR = True

# Filtro de patches.
# Un patch se guarda solo si al menos el 80% de sus píxeles están anotados.
MIN_VALID_RATIO = 0.80

# Clases esperadas en el GeoJSON de QuPath.
CLASS_ORDER = ["BACKGROUND", "no tumor", "Tumor"]
CLASS_TO_ID = {
    "BACKGROUND": 0,
    "no tumor": 1,
    "Tumor": 2,
}
ID_TO_NAME = {v: k for k, v in CLASS_TO_ID.items()}
NUM_CLASSES = len(CLASS_TO_ID)
IGNORE_INDEX = 255

# Unión de ROIs cercanas.
# Déjalo en 0 si no quieres fusionarlas.
MERGE_CLOSE_ROIS_PX = 0

# Split por caso.
SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
N_SPLIT_TRIALS = 1000

# Si quieres comprobar visualmente algún caso:
#   VALIDATE_CASES = "none"   -> no muestra nada
#   VALIDATE_CASES = "first"  -> muestra el primer caso
#   VALIDATE_CASES = "all"    -> muestra todos
#   VALIDATE_CASES = ["B2291CR1_1"] -> muestra solo esos casos
VALIDATE_CASES = "none"
N_PATCHES_TO_SHOW = 6

random.seed(SEED)
np.random.seed(SEED)

print("IMAGES_DIR :", IMAGES_DIR.resolve())
print("GEOJSON_DIR:", GEOJSON_DIR.resolve())
print("EXPORT_DIR :", EXPORT_DIR.resolve())
print("PATCH_SIZE:", PATCH_SIZE)
print("STRIDE    :", STRIDE)


IMAGES_DIR : C:\Users\franc\Downloads\TFM\imagenes
GEOJSON_DIR: C:\Users\franc\Downloads\TFM\geojsons
EXPORT_DIR : C:\Users\franc\Downloads\TFM_UNI_PATCHES_224
PATCH_SIZE: 224
STRIDE    : 112



## 3. Funciones para emparejar imágenes y GeoJSON


In [3]:

def remove_known_suffixes(filename):
    """
    Quita sufijos típicos para emparejar imagen y GeoJSON:
    .ome.tif, .ome.tiff, .tif, .tiff, .geojson
    """
    name = filename.lower()
    for suf in [".ome.tiff", ".ome.tif", ".tiff", ".tif", ".geojson"]:
        if name.endswith(suf):
            return filename[: -len(suf)]
    return Path(filename).stem


def collect_files(folder, extensions):
    folder = Path(folder)
    files = []
    for p in folder.rglob("*"):
        if p.is_file() and any(p.name.lower().endswith(ext) for ext in extensions):
            files.append(p)
    return sorted(files)


def match_images_and_geojsons(images_dir, geojson_dir):
    image_files = collect_files(images_dir, [".ome.tif", ".ome.tiff", ".tif", ".tiff"])
    geojson_files = collect_files(geojson_dir, [".geojson"])

    image_map = {remove_known_suffixes(p.name): p for p in image_files}
    geojson_map = {remove_known_suffixes(p.name): p for p in geojson_files}

    common_keys = sorted(set(image_map.keys()) & set(geojson_map.keys()))
    only_images = sorted(set(image_map.keys()) - set(geojson_map.keys()))
    only_geojsons = sorted(set(geojson_map.keys()) - set(image_map.keys()))

    pairs = []
    for key in common_keys:
        pairs.append({
            "case_id": key,
            "image_path": image_map[key],
            "geojson_path": geojson_map[key],
        })

    print(f"Casos emparejados: {len(pairs)}")

    if only_images:
        print("\nImágenes sin GeoJSON:")
        for x in only_images:
            print(" ", x)

    if only_geojsons:
        print("\nGeoJSON sin imagen:")
        for x in only_geojsons:
            print(" ", x)

    return pairs



## 4. Funciones para leer anotaciones, rasterizar máscaras y extraer ROIs


In [4]:

def load_geojson_features(path):
    with open(path, "r", encoding="utf-8") as f:
        geo = json.load(f)
    return geo.get("features", [])


def update_bbox_from_coords(coords, bbox):
    """
    Actualiza bbox = [minx, miny, maxx, maxy] recorriendo coordenadas anidadas.
    """
    if isinstance(coords[0], (int, float)):
        x, y = coords[:2]
        bbox[0] = min(bbox[0], x)
        bbox[1] = min(bbox[1], y)
        bbox[2] = max(bbox[2], x)
        bbox[3] = max(bbox[3], y)
    else:
        for c in coords:
            update_bbox_from_coords(c, bbox)


def get_global_bbox(features, pad=0):
    """
    Calcula la caja global que contiene todas las anotaciones.
    """
    bbox = [float("inf"), float("inf"), -float("inf"), -float("inf")]

    for feat in features:
        geom = feat.get("geometry", {})
        coords = geom.get("coordinates", None)
        if coords is not None:
            update_bbox_from_coords(coords, bbox)

    minx, miny, maxx, maxy = bbox

    if not np.isfinite([minx, miny, maxx, maxy]).all():
        raise ValueError("No se pudo calcular bbox: GeoJSON sin coordenadas válidas.")

    x0 = math.floor(minx) - pad
    y0 = math.floor(miny) - pad
    x1 = math.ceil(maxx) + pad + 1
    y1 = math.ceil(maxy) + pad + 1

    return x0, y0, x1, y1


def clip_bbox_to_image(img_path, x0, y0, x1, y1):
    """
    Evita leer ventanas fuera de la imagen.
    """
    with rasterio.open(img_path) as src:
        W, H = src.width, src.height

    x0 = max(0, int(x0))
    y0 = max(0, int(y0))
    x1 = min(W, int(x1))
    y1 = min(H, int(y1))

    if x1 <= x0 or y1 <= y0:
        raise ValueError(f"BBox inválido tras recorte: {(x0, y0, x1, y1)}")

    return x0, y0, x1, y1


def read_rgb_window(img_path, x0, y0, x1, y1):
    """
    Lee una ventana RGB del OME-TIFF/TIFF.
    Si la imagen tiene una sola banda, la replica a 3 canales.
    """
    w = x1 - x0
    h = y1 - y0

    with rasterio.open(img_path) as src:
        window = Window(x0, y0, w, h)

        if src.count >= 3:
            arr = src.read([1, 2, 3], window=window)
            arr = np.moveaxis(arr, 0, -1)
        elif src.count == 1:
            band = src.read(1, window=window)
            arr = np.stack([band, band, band], axis=-1)
        else:
            raise ValueError(f"La imagen no tiene bandas válidas: {img_path}")

    if arr.dtype != np.uint8:
        arr = arr.astype(np.float32)
        if arr.max() <= 1.0:
            arr = arr * 255.0
        arr = np.clip(arr, 0, 255).astype(np.uint8)

    return arr


def get_feature_class_name(feature):
    """
    Extrae el nombre de clase del GeoJSON exportado desde QuPath.
    Estructura esperada:
        properties -> classification -> name
    """
    props = feature.get("properties", {})
    classification = props.get("classification", {})

    if isinstance(classification, dict) and "name" in classification:
        return classification["name"]

    # Fallback por si el GeoJSON tiene otro formato.
    if "name" in props:
        return props["name"]

    return None


def shift_ring_to_local(ring, x_offset, y_offset):
    """
    Pasa coordenadas globales de QuPath a coordenadas locales de la ventana leída.
    """
    pts = []
    for pt in ring:
        x, y = pt[:2]
        pts.append((int(round(x - x_offset)), int(round(y - y_offset))))
    return pts


def rasterize_polygon_with_holes(polygon_coords, width, height, x_offset, y_offset):
    """
    Rasteriza un Polygon de GeoJSON con posible anillo exterior y agujeros.
    polygon_coords = [outer_ring, hole1, hole2, ...]
    """
    mask_img = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask_img)

    if len(polygon_coords) == 0:
        return np.zeros((height, width), dtype=bool)

    outer = shift_ring_to_local(polygon_coords[0], x_offset, y_offset)
    draw.polygon(outer, fill=1)

    for hole in polygon_coords[1:]:
        hole_pts = shift_ring_to_local(hole, x_offset, y_offset)
        draw.polygon(hole_pts, fill=0)

    return np.array(mask_img, dtype=bool)


def feature_to_mask(feature, width, height, x_offset, y_offset):
    geom = feature.get("geometry", {})
    geom_type = geom.get("type")
    coords = geom.get("coordinates")

    mask = np.zeros((height, width), dtype=bool)

    if coords is None:
        return mask

    if geom_type == "Polygon":
        mask |= rasterize_polygon_with_holes(coords, width, height, x_offset, y_offset)

    elif geom_type == "MultiPolygon":
        for polygon in coords:
            mask |= rasterize_polygon_with_holes(polygon, width, height, x_offset, y_offset)

    else:
        raise ValueError(f"Geometría no soportada: {geom_type}")

    return mask


def build_class_masks(features, x0, y0, x1, y1):
    """
    Crea una máscara booleana por clase.
    """
    w = x1 - x0
    h = y1 - y0

    masks = {cls: np.zeros((h, w), dtype=bool) for cls in CLASS_ORDER}
    unknown_classes = set()

    for feat in features:
        cls = get_feature_class_name(feat)

        if cls not in CLASS_TO_ID:
            unknown_classes.add(cls)
            continue

        feat_mask = feature_to_mask(feat, w, h, x0, y0)
        masks[cls] |= feat_mask

    if unknown_classes:
        print("  Aviso: clases ignoradas en GeoJSON:", sorted([str(x) for x in unknown_classes]))

    return masks


def build_semantic_mask(masks, ignore_index=255):
    """
    Construye la máscara multiclase final.

    Valores:
        0   = BACKGROUND
        1   = no tumor
        2   = Tumor
        255 = ignore_index / no anotado

    Prioridad usada:
        - no tumor y tumor se pintan solo fuera del background.
        - background se pinta al final como 0.
    """
    h, w = next(iter(masks.values())).shape
    semantic_mask = np.full((h, w), ignore_index, dtype=np.uint8)

    background = masks["BACKGROUND"]
    no_tumor = masks["no tumor"] & (~background)
    tumor = masks["Tumor"] & (~background)

    semantic_mask[no_tumor] = CLASS_TO_ID["no tumor"]
    semantic_mask[tumor] = CLASS_TO_ID["Tumor"]
    semantic_mask[background] = CLASS_TO_ID["BACKGROUND"]

    return semantic_mask


def find_connected_rois(semantic_mask, merge_close_rois_px=0):
    """
    Devuelve cajas (x0, y0, x1, y1) de regiones conectadas anotadas.
    """
    valid = semantic_mask != IGNORE_INDEX

    if merge_close_rois_px and merge_close_rois_px > 0:
        structure = np.ones((merge_close_rois_px, merge_close_rois_px), dtype=bool)
        valid = ndimage.binary_dilation(valid, structure=structure)

    labeled, n = ndimage.label(valid)
    objects = ndimage.find_objects(labeled)

    boxes = []
    for sl in objects:
        if sl is None:
            continue
        ys, xs = sl
        y0, y1 = ys.start, ys.stop
        x0, x1 = xs.start, xs.stop
        boxes.append((x0, y0, x1, y1))

    return boxes


def crop_rois(image_global, mask_global, roi_boxes):
    rois = []
    for i, (x0, y0, x1, y1) in enumerate(roi_boxes):
        rois.append({
            "roi_id": i,
            "bbox_local": (x0, y0, x1, y1),
            "image": image_global[y0:y1, x0:x1],
            "mask": mask_global[y0:y1, x0:x1],
        })
    return rois



## 5. Funciones para extraer patches y procesar todos los casos


In [5]:

def sliding_positions(length, patch_size, stride):
    if length <= patch_size:
        return [0]

    positions = list(range(0, length - patch_size + 1, stride))
    if positions[-1] != length - patch_size:
        positions.append(length - patch_size)

    return positions


def pad_to_min_size(image, mask, min_h, min_w, ignore_index=255):
    h, w = mask.shape

    pad_h = max(0, min_h - h)
    pad_w = max(0, min_w - w)

    if pad_h == 0 and pad_w == 0:
        return image, mask

    image_pad_mode = "reflect" if h > 1 and w > 1 else "edge"

    image = np.pad(
        image,
        ((0, pad_h), (0, pad_w), (0, 0)),
        mode=image_pad_mode,
    )

    mask = np.pad(
        mask,
        ((0, pad_h), (0, pad_w)),
        mode="constant",
        constant_values=ignore_index,
    )

    return image, mask


def extract_patches_from_roi(
    image,
    mask,
    patch_size=256,
    stride=128,
    ignore_index=255,
    min_valid_ratio=0.8,
):
    image, mask = pad_to_min_size(image, mask, patch_size, patch_size, ignore_index)

    h, w = mask.shape
    ys = sliding_positions(h, patch_size, stride)
    xs = sliding_positions(w, patch_size, stride)

    patches = []

    for y in ys:
        for x in xs:
            img_patch = image[y:y + patch_size, x:x + patch_size]
            mask_patch = mask[y:y + patch_size, x:x + patch_size]

            valid_ratio = np.mean(mask_patch != ignore_index)

            if valid_ratio >= min_valid_ratio:
                patches.append((img_patch, mask_patch))

    return patches


def should_validate_case(case_id, case_index, mode):
    if mode is None or mode == "none":
        return False
    if mode == "all":
        return True
    if mode == "first":
        return case_index == 0
    if isinstance(mode, (list, tuple, set)):
        return case_id in mode
    return False


def show_random_patches_from_case(case_result, n=6):
    patches = case_result["patches"]
    if len(patches) == 0:
        print("No hay patches para mostrar.")
        return

    idxs = np.random.choice(len(patches), size=min(n, len(patches)), replace=False)

    for idx in idxs:
        p = patches[idx]
        fig, axes = plt.subplots(1, 2, figsize=(7, 3))
        axes[0].imshow(p["image"])
        axes[0].set_title(f"{p['case_id']} | ROI {p['roi_id']} | patch {p['patch_id']}")
        axes[0].axis("off")

        axes[1].imshow(p["mask"], vmin=0, vmax=2)
        axes[1].set_title("Máscara")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()


def process_case(case):
    case_id = case["case_id"]
    img_path = case["image_path"]
    geojson_path = case["geojson_path"]

    print("=" * 80)
    print(f"Procesando caso: {case_id}")
    print(f"Imagen : {img_path.name}")
    print(f"GeoJSON: {geojson_path.name}")

    features = load_geojson_features(geojson_path)
    if len(features) == 0:
        print("  -> Sin features. Se salta.")
        return None

    x0, y0, x1, y1 = get_global_bbox(features, pad=0)
    x0, y0, x1, y1 = clip_bbox_to_image(img_path, x0, y0, x1, y1)

    image_global = read_rgb_window(img_path, x0, y0, x1, y1)
    class_masks = build_class_masks(features, x0, y0, x1, y1)
    semantic_mask = build_semantic_mask(class_masks, ignore_index=IGNORE_INDEX)

    roi_boxes = find_connected_rois(
        semantic_mask,
        merge_close_rois_px=MERGE_CLOSE_ROIS_PX,
    )
    rois = crop_rois(image_global, semantic_mask, roi_boxes)

    case_patches = []

    for roi in rois:
        roi_patches = extract_patches_from_roi(
            roi["image"],
            roi["mask"],
            patch_size=PATCH_SIZE,
            stride=STRIDE,
            ignore_index=IGNORE_INDEX,
            min_valid_ratio=MIN_VALID_RATIO,
        )

        for patch_idx, (img_patch, mask_patch) in enumerate(roi_patches):
            case_patches.append({
                "case_id": case_id,
                "roi_id": roi["roi_id"],
                "patch_id": patch_idx,
                "image": img_patch,
                "mask": mask_patch,
            })

        print(f"  ROI {roi['roi_id']}: {len(roi_patches)} patches")

    print(f"  Total ROIs   : {len(rois)}")
    print(f"  Total patches: {len(case_patches)}")

    return {
        "case_id": case_id,
        "image_path": img_path,
        "geojson_path": geojson_path,
        "bbox_global": (x0, y0, x1, y1),
        "image_global": image_global,
        "semantic_mask": semantic_mask,
        "roi_boxes": roi_boxes,
        "rois": rois,
        "patches": case_patches,
    }


def process_all_cases(images_dir=IMAGES_DIR, geojson_dir=GEOJSON_DIR, validate_cases=VALIDATE_CASES):
    print("1) Emparejando imágenes y GeoJSON...")
    pairs = match_images_and_geojsons(images_dir, geojson_dir)

    if len(pairs) == 0:
        raise RuntimeError(
            f"No se han encontrado pares imagen-GeoJSON en:\n"
            f"  imágenes: {images_dir}\n"
            f"  geojsons: {geojson_dir}"
        )

    print("\n2) Procesando casos...")
    all_case_data = []
    all_patches_meta = []

    for i, case in enumerate(pairs):
        result = process_case(case)

        if result is None:
            continue

        all_case_data.append(result)
        all_patches_meta.extend(result["patches"])

        if should_validate_case(result["case_id"], i, validate_cases):
            show_random_patches_from_case(result, n=N_PATCHES_TO_SHOW)

    print("\n" + "=" * 80)
    print("RESUMEN GLOBAL")
    print("=" * 80)
    print(f"Casos procesados: {len(all_case_data)}")
    print(f"Total patches   : {len(all_patches_meta)}")

    if len(all_patches_meta) == 0:
        raise RuntimeError("No se ha generado ningún patch.")

    all_patches = [(x["image"], x["mask"]) for x in all_patches_meta]

    return all_case_data, all_patches_meta, all_patches



## 6. Funciones para split por caso y resumen de distribución


In [6]:

def unpack_patch_for_counts(patch):
    if isinstance(patch, dict):
        return patch["image"], patch["mask"]
    return patch


def count_pixels_in_patches(patches, num_classes=3, ignore_index=255):
    counts = np.zeros(num_classes, dtype=np.int64)
    ignored = 0

    for patch in patches:
        _, mask = unpack_patch_for_counts(patch)
        mask = np.asarray(mask)

        ignored += int(np.sum(mask == ignore_index))

        valid = mask != ignore_index
        valid_mask = mask[valid]

        if valid_mask.size > 0:
            counts += np.bincount(
                valid_mask.reshape(-1),
                minlength=num_classes,
            )[:num_classes]

    return counts, ignored


def class_distribution_from_counts(counts):
    total = counts.sum()
    if total == 0:
        return np.zeros_like(counts, dtype=np.float64)
    return counts / total


def case_class_counts(case_data, num_classes=3, ignore_index=255):
    patches = case_data["patches"]
    counts, ignored = count_pixels_in_patches(
        patches,
        num_classes=num_classes,
        ignore_index=ignore_index,
    )
    return counts, ignored


def get_group_id(case_id):
    """
    Elimina sufijos finales tipo _1, _2, _3...
    Así B2291CR1_1 y B2291CR1_3 van juntos al mismo split.
    """
    return re.sub(r"_\d+$", "", str(case_id))


def get_patches_from_cases(all_case_data, case_ids):
    case_ids = set(case_ids)
    patches = []
    for case_data in all_case_data:
        if case_data["case_id"] in case_ids:
            patches.extend(case_data["patches"])
    return patches


def split_case_ids_balanced_random_search(
    case_data_list,
    train_ratio=0.70,
    val_ratio=0.15,
    seed=42,
    n_trials=1000,
    num_classes=3,
    ignore_index=255,
):
    """
    Split equilibrado por grupo base.

    Ejemplo:
        B2291CR1_1 y B2291CR1_3 se consideran el mismo grupo B2291CR1
        y siempre van juntos al mismo split.
    """
    rng = random.Random(seed)

    group_to_case_ids = {}
    for case_data in case_data_list:
        case_id = case_data["case_id"]
        group_id = get_group_id(case_id)
        group_to_case_ids.setdefault(group_id, []).append(case_id)

    group_ids = list(group_to_case_ids.keys())
    n_groups = len(group_ids)

    if n_groups < 3:
        raise RuntimeError("Necesitas al menos 3 grupos para hacer train/val/test.")

    n_train = max(1, int(round(n_groups * train_ratio)))
    n_val = max(1, int(round(n_groups * val_ratio)))

    if n_train + n_val >= n_groups:
        n_train = max(1, n_groups - 2)
        n_val = 1

    case_data_by_id = {case_data["case_id"]: case_data for case_data in case_data_list}

    counts_by_group = {}
    for group_id, case_ids in group_to_case_ids.items():
        group_counts = np.zeros(num_classes, dtype=np.int64)
        for case_id in case_ids:
            counts, _ = case_class_counts(
                case_data_by_id[case_id],
                num_classes=num_classes,
                ignore_index=ignore_index,
            )
            group_counts += counts
        counts_by_group[group_id] = group_counts

    global_counts = np.sum(
        np.stack([counts_by_group[group_id] for group_id in group_ids], axis=0),
        axis=0,
    )
    global_dist = class_distribution_from_counts(global_counts)

    def counts_for_groups(selected_group_ids):
        if len(selected_group_ids) == 0:
            return np.zeros(num_classes, dtype=np.int64)
        return np.sum(
            np.stack([counts_by_group[group_id] for group_id in selected_group_ids], axis=0),
            axis=0,
        )

    def score_split(train_group_ids, val_group_ids, test_group_ids):
        train_counts = counts_for_groups(train_group_ids)
        val_counts = counts_for_groups(val_group_ids)
        test_counts = counts_for_groups(test_group_ids)

        train_dist = class_distribution_from_counts(train_counts)
        val_dist = class_distribution_from_counts(val_counts)
        test_dist = class_distribution_from_counts(test_counts)

        train_score = np.abs(train_dist - global_dist).sum()
        val_score = np.abs(val_dist - global_dist).sum()
        test_score = np.abs(test_dist - global_dist).sum()

        zero_penalty = 0.0
        for counts in [train_counts, val_counts, test_counts]:
            for cls_id in range(num_classes):
                if counts[cls_id] == 0:
                    zero_penalty += 10.0

        return train_score + 2.0 * val_score + 2.0 * test_score + zero_penalty

    best_score = float("inf")
    best_split = None

    for _ in range(n_trials):
        shuffled_groups = group_ids.copy()
        rng.shuffle(shuffled_groups)

        train_group_ids = shuffled_groups[:n_train]
        val_group_ids = shuffled_groups[n_train:n_train + n_val]
        test_group_ids = shuffled_groups[n_train + n_val:]

        score = score_split(train_group_ids, val_group_ids, test_group_ids)

        if score < best_score:
            best_score = score
            best_split = (train_group_ids, val_group_ids, test_group_ids)

    train_group_ids, val_group_ids, test_group_ids = best_split

    train_case_ids = [
        case_id
        for group_id in train_group_ids
        for case_id in group_to_case_ids[group_id]
    ]
    val_case_ids = [
        case_id
        for group_id in val_group_ids
        for case_id in group_to_case_ids[group_id]
    ]
    test_case_ids = [
        case_id
        for group_id in test_group_ids
        for case_id in group_to_case_ids[group_id]
    ]

    print("\nMejor score de split equilibrado por grupos:", best_score)
    print("\nGrupos TRAIN:", train_group_ids)
    print("Grupos VAL  :", val_group_ids)
    print("Grupos TEST :", test_group_ids)

    print("\nNº escenas train:", len(train_case_ids))
    print("Nº escenas val  :", len(val_case_ids))
    print("Nº escenas test :", len(test_case_ids))

    return train_case_ids, val_case_ids, test_case_ids


def check_no_case_leakage(train_case_ids, val_case_ids, test_case_ids):
    train_set = set(train_case_ids)
    val_set = set(val_case_ids)
    test_set = set(test_case_ids)

    train_val = train_set & val_set
    train_test = train_set & test_set
    val_test = val_set & test_set

    print("\n" + "=" * 80)
    print("COMPROBACIÓN DE FUGA DE CASOS")
    print("=" * 80)
    print("Train ∩ Val :", train_val)
    print("Train ∩ Test:", train_test)
    print("Val ∩ Test  :", val_test)

    if len(train_val) > 0 or len(train_test) > 0 or len(val_test) > 0:
        raise RuntimeError("Hay fuga de datos: algún caso aparece en más de un split.")

    print("OK: no hay fuga de casos.")


def split_distribution_report(split_name, patches, num_classes=3, ignore_index=255):
    counts, ignored = count_pixels_in_patches(
        patches,
        num_classes=num_classes,
        ignore_index=ignore_index,
    )
    proportions = class_distribution_from_counts(counts)

    row = {
        "split": split_name,
        "n_patches": len(patches),
        "valid_pixels": int(counts.sum()),
        "ignored_pixels": int(ignored),
    }

    for cls_id in range(num_classes):
        class_name = ID_TO_NAME.get(cls_id, str(cls_id))
        row[f"pixels_{class_name}"] = int(counts[cls_id])
        row[f"pct_{class_name}"] = float(proportions[cls_id])

    return row



## 7. Funciones para guardar los patches como tu estructura


In [7]:

def safe_name(text):
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9_\-]+", "_", text)
    return text.strip("_")


def get_patch_image_mask_and_meta(patch, fallback_idx):
    if isinstance(patch, dict):
        image = patch["image"]
        mask = patch["mask"]
        case_id = patch.get("case_id", "unknown_case")
        roi_id = patch.get("roi_id", "unknown_roi")
        patch_id = patch.get("patch_id", fallback_idx)
    else:
        image, mask = patch
        case_id = "unknown_case"
        roi_id = "unknown_roi"
        patch_id = fallback_idx

    return image, mask, case_id, roi_id, patch_id


def save_patch_split(patches, split_name, export_dir):
    """
    Guarda los patches de un split en:
        export_dir/split_name/images
        export_dir/split_name/masks

    Devuelve filas para metadata.csv.
    """
    image_dir = export_dir / split_name / "images"
    mask_dir = export_dir / split_name / "masks"

    image_dir.mkdir(parents=True, exist_ok=True)
    mask_dir.mkdir(parents=True, exist_ok=True)

    rows = []

    for i, patch in enumerate(tqdm(patches, desc=f"Guardando {split_name}")):
        image, mask, case_id, roi_id, patch_id = get_patch_image_mask_and_meta(
            patch,
            fallback_idx=i,
        )

        image = np.asarray(image)
        mask = np.asarray(mask)

        # Imagen RGB.
        if image.dtype != np.uint8:
            image = image.astype(np.float32)
            if image.max() <= 1.0:
                image = image * 255.0
            image = np.clip(image, 0, 255).astype(np.uint8)

        # Asegurar imagen de 3 canales.
        if image.ndim == 2:
            image = np.stack([image, image, image], axis=-1)
        elif image.ndim == 3 and image.shape[-1] > 3:
            image = image[..., :3]

        # Máscara con clases 0, 1, 2 y 255 para ignore_index.
        if mask.ndim == 3:
            mask = mask[..., 0]
        if mask.dtype != np.uint8:
            mask = mask.astype(np.uint8)

        case_name = safe_name(case_id)
        roi_name = safe_name(roi_id)
        patch_name = safe_name(patch_id)

        filename_base = f"{split_name}_{case_name}_roi{roi_name}_patch{patch_name}_{i:06d}"

        image_path = image_dir / f"{filename_base}.png"
        mask_path = mask_dir / f"{filename_base}.png"

        Image.fromarray(image).save(image_path)
        Image.fromarray(mask).save(mask_path)

        rows.append({
            "split": split_name,
            "case_id": case_id,
            "roi_id": roi_id,
            "patch_id": patch_id,
            "image_path": str(image_path.relative_to(export_dir)).replace("\\", "/"),
            "mask_path": str(mask_path.relative_to(export_dir)).replace("\\", "/"),
        })

    return rows


def prepare_export_dir(export_dir, overwrite=True):
    export_dir = Path(export_dir)

    if export_dir.exists() and overwrite:
        print(f"Borrando carpeta anterior: {export_dir}")
        shutil.rmtree(export_dir)

    export_dir.mkdir(parents=True, exist_ok=True)
    return export_dir



## 8. Ejecutar procesamiento y guardar dataset

Ejecuta esta celda para generar todo el dataset de patches.


In [8]:

# ============================================================
# EJECUCIÓN PRINCIPAL
# ============================================================

# 1. Procesar todos los casos y generar patches en memoria.
all_case_data, all_patches_meta, all_patches = process_all_cases(
    images_dir=IMAGES_DIR,
    geojson_dir=GEOJSON_DIR,
    validate_cases=VALIDATE_CASES,
)

# 2. Split por caso, igual que en tu entrenamiento actual.
train_case_ids, val_case_ids, test_case_ids = split_case_ids_balanced_random_search(
    all_case_data,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    seed=SEED,
    n_trials=N_SPLIT_TRIALS,
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
)

check_no_case_leakage(train_case_ids, val_case_ids, test_case_ids)

# 3. Sacar patches de cada split.
train_patches = get_patches_from_cases(all_case_data, train_case_ids)
val_patches = get_patches_from_cases(all_case_data, val_case_ids)
test_patches = get_patches_from_cases(all_case_data, test_case_ids)

print("\n" + "=" * 80)
print("NÚMERO DE PATCHES POR SPLIT")
print("=" * 80)
print("Train patches:", len(train_patches))
print("Val patches  :", len(val_patches))
print("Test patches :", len(test_patches))

# 4. Preparar carpeta de salida.
EXPORT_DIR = prepare_export_dir(EXPORT_DIR, overwrite=OVERWRITE_EXPORT_DIR)

# 5. Exportar patches.
all_rows = []
all_rows += save_patch_split(train_patches, "train", EXPORT_DIR)
all_rows += save_patch_split(val_patches, "val", EXPORT_DIR)
all_rows += save_patch_split(test_patches, "test", EXPORT_DIR)

metadata_df = pd.DataFrame(all_rows)
metadata_df.to_csv(EXPORT_DIR / "metadata.csv", index=False)

# 6. Guardar IDs de caso para poder reproducir el split.
split_info = {
    "seed": SEED,
    "patch_size": PATCH_SIZE,
    "stride": STRIDE,
    "min_valid_ratio": MIN_VALID_RATIO,
    "train_case_ids": list(train_case_ids),
    "val_case_ids": list(val_case_ids),
    "test_case_ids": list(test_case_ids),
}

with open(EXPORT_DIR / "split_case_ids.json", "w", encoding="utf-8") as f:
    json.dump(split_info, f, indent=4, ensure_ascii=False)

# 7. Guardar resumen de distribución por split.
split_summary_df = pd.DataFrame([
    split_distribution_report("train", train_patches, NUM_CLASSES, IGNORE_INDEX),
    split_distribution_report("val", val_patches, NUM_CLASSES, IGNORE_INDEX),
    split_distribution_report("test", test_patches, NUM_CLASSES, IGNORE_INDEX),
])
split_summary_df.to_csv(EXPORT_DIR / "split_distribution.csv", index=False)

print("\n" + "=" * 80)
print("DATASET EXPORTADO")
print("=" * 80)
print("Carpeta:", EXPORT_DIR.resolve())
print("metadata.csv:", (EXPORT_DIR / "metadata.csv").resolve())
print("split_case_ids.json:", (EXPORT_DIR / "split_case_ids.json").resolve())
print("split_distribution.csv:", (EXPORT_DIR / "split_distribution.csv").resolve())

print("\nPrimeras filas de metadata.csv:")
display(metadata_df.head())

print("\nDistribución por split:")
display(split_summary_df)


1) Emparejando imágenes y GeoJSON...
Casos emparejados: 15

Imágenes sin GeoJSON:
  B2289CR1
  B2293CR2

2) Procesando casos...
Procesando caso: B21122CR6
Imagen : B21122CR6.ome.tif
GeoJSON: B21122CR6.geojson


C:\Users\franc\anaconda3\envs\segenv\Lib\site-packages\rasterio\__init__.py:368: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


  ROI 0: 1286 patches
  ROI 1: 1274 patches
  Total ROIs   : 2
  Total patches: 2560
Procesando caso: B2249CR2
Imagen : B2249CR2.ome.tif
GeoJSON: B2249CR2.geojson
  ROI 0: 756 patches
  ROI 1: 1841 patches
  ROI 2: 800 patches
  Total ROIs   : 3
  Total patches: 3397
Procesando caso: B2275S15
Imagen : B2275S15.ome.tif
GeoJSON: B2275S15.geojson
  ROI 0: 852 patches
  ROI 1: 686 patches
  Total ROIs   : 2
  Total patches: 1538
Procesando caso: B2277CR3_1
Imagen : B2277CR3_1.ome.tif
GeoJSON: B2277CR3_1.geojson
  ROI 0: 2793 patches
  Total ROIs   : 1
  Total patches: 2793
Procesando caso: B2279CR1
Imagen : B2279CR1.ome.tif
GeoJSON: B2279CR1.geojson
  ROI 0: 3024 patches
  ROI 1: 2990 patches
  Total ROIs   : 2
  Total patches: 6014
Procesando caso: B2290CR1
Imagen : B2290CR1.ome.tif
GeoJSON: B2290CR1.geojson
  ROI 0: 1886 patches
  Total ROIs   : 1
  Total patches: 1886
Procesando caso: B2291CR1_1
Imagen : B2291CR1_1.ome.tif
GeoJSON: B2291CR1_1.geojson
  ROI 0: 156 patches
  ROI 1: 1522 p

Guardando test: 100%|██████████| 5030/5030 [01:52<00:00, 44.53it/s]



DATASET EXPORTADO
Carpeta: C:\Users\franc\Downloads\TFM_UNI_PATCHES_224
metadata.csv: C:\Users\franc\Downloads\TFM_UNI_PATCHES_224\metadata.csv
split_case_ids.json: C:\Users\franc\Downloads\TFM_UNI_PATCHES_224\split_case_ids.json
split_distribution.csv: C:\Users\franc\Downloads\TFM_UNI_PATCHES_224\split_distribution.csv

Primeras filas de metadata.csv:


,split,case_id,roi_id,patch_id,image_path,mask_path
0,train,B2249CR2,0,0,train/images/train_B2249CR2_roi0_patch0_000000...,train/masks/train_B2249CR2_roi0_patch0_000000.png
1,train,B2249CR2,0,1,train/images/train_B2249CR2_roi0_patch1_000001...,train/masks/train_B2249CR2_roi0_patch1_000001.png
2,train,B2249CR2,0,2,train/images/train_B2249CR2_roi0_patch2_000002...,train/masks/train_B2249CR2_roi0_patch2_000002.png
3,train,B2249CR2,0,3,train/images/train_B2249CR2_roi0_patch3_000003...,train/masks/train_B2249CR2_roi0_patch3_000003.png
4,train,B2249CR2,0,4,train/images/train_B2249CR2_roi0_patch4_000004...,train/masks/train_B2249CR2_roi0_patch4_000004.png



Distribución por split:


,split,n_patches,valid_pixels,ignored_pixels,pixels_BACKGROUND,pct_BACKGROUND,pixels_no tumor,pct_no tumor,pixels_Tumor,pct_Tumor
0,train,32390,1621460227,3740413,575347703,0.354833,639838367,0.394606,406274157,0.250561
1,val,8807,441853416,46616,148373758,0.335799,169978712,0.384695,123500946,0.279507
2,test,5030,251237050,1148230,85560443,0.340557,94233761,0.375079,71442846,0.284364
